# Lecture 23 — Model Deployment

**CMU 10-414/714, Deep Learning Systems (lecture by Tianqi Chen)**
Companion notebook · Sangram Lembe

Training happens in a flexible Python framework; deployment happens on a phone, a browser or a small board,
often with no Python at all. The standard route: **export the computational graph** to a portable format,
then run it with a small **inference engine** that plans memory and fuses operators. This notebook does all
four steps:

1. train a model with the autodiff engine
2. export its graph to plain JSON — no framework needed to read it
3. run it with a tiny NumPy-only inference engine, with memory reuse
4. fuse operators, and see the one thing a traced graph cannot capture

In [1]:
import numpy as np

class Tensor:
    """A needle-style tensor: data plus the op and inputs that produced it."""

    def __init__(self, array, requires_grad=True):
        self._init(None, [], np.asarray(array, dtype=np.float64), requires_grad)

    def _init(self, op, inputs, cached, requires_grad):
        self.op, self.inputs = op, list(inputs)
        self.cached, self.requires_grad, self.grad = cached, requires_grad, None

    @classmethod
    def make_const(cls, array):
        """A leaf with no history - what detach() returns."""
        t = Tensor.__new__(Tensor)
        t._init(None, [], array, False)
        return t

    @staticmethod
    def make_from_op(op, inputs):
        out = op.compute(*[i.cached for i in inputs])
        if not any(i.requires_grad for i in inputs):
            return Tensor.make_const(out)        # nothing needs a gradient: no graph
        t = Tensor.__new__(Tensor)
        t._init(op, inputs, out, True)
        return t

    # .data is a detached view that SHARES memory; assigning to it updates in place
    @property
    def data(self):
        return Tensor.make_const(self.cached)
    @data.setter
    def data(self, value):
        self.cached = value.cached if isinstance(value, Tensor) else np.asarray(value)

    def detach(self):  return self.data
    def numpy(self):   return self.cached
    shape = property(lambda s: s.cached.shape)
    def __repr__(self):
        return f"Tensor({np.array2string(self.cached, precision=4)}, op={type(self.op).__name__ if self.op else None})"

    def __add__(s, o):  return EAdd()(s, o) if isinstance(o, Tensor) else AddS(o)(s)
    def __mul__(s, o):  return EMul()(s, o) if isinstance(o, Tensor) else MulS(o)(s)
    def __neg__(s):     return MulS(-1.0)(s)
    def __sub__(s, o):  return s + (-o)
    def __rsub__(s, o): return (-s) + o
    def __pow__(s, c):  return PowS(c)(s)
    def __truediv__(s, o):
        return s * (o ** -1.0) if isinstance(o, Tensor) else MulS(1.0 / o)(s)
    def __matmul__(s, o): return MatMul()(s, o)
    __radd__, __rmul__ = __add__, __mul__

    def backward(self):
        backprop(self, Tensor.make_const(np.ones_like(self.cached)))


class Op:
    def __call__(self, *ins):
        return Tensor.make_from_op(self, ins)
    def grads(self, g, node):
        r = self.gradient(g, node)
        return r if isinstance(r, tuple) else (r,)

class EAdd(Op):
    def compute(s, a, b): return a + b
    def gradient(s, g, n): return g, g
class AddS(Op):
    def __init__(s, c): s.c = c
    def compute(s, a): return a + s.c
    def gradient(s, g, n): return g
class EMul(Op):
    def compute(s, a, b): return a * b
    def gradient(s, g, n): return g * n.inputs[1], g * n.inputs[0]
class MulS(Op):
    def __init__(s, c): s.c = c
    def compute(s, a): return a * s.c
    def gradient(s, g, n): return g * s.c
class PowS(Op):
    def __init__(s, c): s.c = c
    def compute(s, a): return a ** s.c
    def gradient(s, g, n): return g * (n.inputs[0] ** (s.c - 1)) * s.c
class MatMul(Op):
    def compute(s, a, b): return a @ b
    def gradient(s, g, n):
        return g @ transpose(n.inputs[1]), transpose(n.inputs[0]) @ g
class Transpose(Op):
    def compute(s, a): return a.T
    def gradient(s, g, n): return transpose(g)
class Reshape(Op):
    def __init__(s, shape): s.shape = shape
    def compute(s, a): return a.reshape(s.shape)
    def gradient(s, g, n): return reshape(g, n.inputs[0].shape)
class BroadcastTo(Op):
    def __init__(s, shape): s.shape = shape
    def compute(s, a): return np.broadcast_to(a, s.shape).copy()
    def gradient(s, g, n):                        # broadcast forward = sum backward
        ins = n.inputs[0].shape
        pad = len(s.shape) - len(ins)
        axes = tuple(range(pad)) + tuple(pad + i for i, d in enumerate(ins)
                                         if d == 1 and s.shape[pad + i] != 1)
        return reshape(summation(g, axes) if axes else g, ins)
class Summation(Op):
    def __init__(s, axes=None):
        s.axes = (axes,) if isinstance(axes, int) else axes
    def compute(s, a): return a.sum(axis=s.axes)
    def gradient(s, g, n):
        ins = n.inputs[0].shape
        ax = range(len(ins)) if s.axes is None else s.axes
        keep = tuple(1 if i in ax else d for i, d in enumerate(ins))
        return broadcast_to(reshape(g, keep), ins)
class ReLUOp(Op):
    def compute(s, a): return np.maximum(0, a)
    def gradient(s, g, n):
        return g * Tensor.make_const((n.inputs[0].cached > 0) * 1.0)
class Exp(Op):
    def compute(s, a): return np.exp(a)
    def gradient(s, g, n): return g * exp(n.inputs[0])
class LogSumExp(Op):
    """Stable log-sum-exp over axis 1: subtract the row max first."""
    def compute(s, a):
        m = a.max(1, keepdims=True)
        return (np.log(np.exp(a - m).sum(1, keepdims=True)) + m)[:, 0]
    def gradient(s, g, n):
        z = n.inputs[0]
        sm = exp(z - broadcast_to(reshape(n, (z.shape[0], 1)), z.shape))
        return broadcast_to(reshape(g, (z.shape[0], 1)), z.shape) * sm

transpose    = lambda a: Transpose()(a)
reshape      = lambda a, s: Reshape(s)(a)
broadcast_to = lambda a, s: BroadcastTo(s)(a)
summation    = lambda a, axes=None: Summation(axes)(a)
relu         = lambda a: ReLUOp()(a)
exp          = lambda a: Exp()(a)
logsumexp    = lambda a: LogSumExp()(a)

def topo_sort(root):
    """Post-order DFS, iterative so that very deep graphs cannot overflow the stack."""
    order, seen, stack = [], set(), [(root, False)]
    while stack:
        node, done = stack.pop()
        if done:
            order.append(node)
            continue
        if id(node) in seen:
            continue
        seen.add(id(node))
        stack.append((node, True))
        for i in node.inputs:
            if id(i) not in seen:
                stack.append((i, False))
    return order

def backprop(out, seed):
    """Reverse-mode AD: walk reverse topological order, summing partial adjoints."""
    parts = {id(out): [seed]}
    for node in reversed(topo_sort(out)):
        if id(node) not in parts: continue
        g = parts[id(node)][0]
        for extra in parts[id(node)][1:]:
            g = g + extra
        node.grad = g
        if node.op is not None:
            for inp, p in zip(node.inputs, node.op.grads(g, node)):
                if inp.requires_grad:
                    parts.setdefault(id(inp), []).append(p)

def numeric_grad(f, param, eps=1e-6):
    """Two-sided finite differences of scalar f() w.r.t. every entry of param."""
    g = np.zeros_like(param.cached)
    for idx in np.ndindex(*param.cached.shape):
        old = param.cached[idx]
        param.cached[idx] = old + eps; a = float(f().cached)
        param.cached[idx] = old - eps; b = float(f().cached)
        param.cached[idx] = old
        g[idx] = (a - b) / (2 * eps)
    return g



import json
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(digits.data / 16.0, digits.target, test_size=0.25,
                                      random_state=0, stratify=digits.target)
rng = np.random.default_rng(0)
print("engine and data ready")

engine and data ready


## 1. Train a small model

In [2]:
W1, b1 = Tensor(rng.normal(0, np.sqrt(2 / 64), (64, 64))), Tensor(np.zeros((1, 64)))
W2, b2 = Tensor(rng.normal(0, np.sqrt(2 / 64), (64, 10))), Tensor(np.zeros((1, 10)))
params = [W1, b1, W2, b2]

def model(x):
    h = x @ W1
    h = relu(h + broadcast_to(b1, h.shape))
    out = h @ W2
    return out + broadcast_to(b2, out.shape)

for epoch in range(30):
    for i in range(0, len(Xtr), 64):
        xb = Tensor(Xtr[i:i + 64], requires_grad=False)
        Y = Tensor.make_const(np.eye(10)[ytr[i:i + 64]])
        logits = model(xb)
        loss = summation(logsumexp(logits) - summation(logits * Y, 1)) / xb.shape[0]
        loss.backward()
        for p in params:
            p.data = p.cached - 0.3 * p.grad.cached
            p.grad = None

framework_pred = model(Tensor(Xte, requires_grad=False)).cached
print("test accuracy in the framework:", np.mean(framework_pred.argmax(1) == yte))

test accuracy in the framework: 0.9711111111111111


## 2. Export the computational graph

Run the model once on a placeholder input, walk the graph in topological order, and write every node out as
plain data: its operation, its inputs, and the weights as constants. Explicit `broadcast_to` nodes become
no-ops, since the runtime broadcasts implicitly.

In [3]:
def export(model, n_features, placeholder=None):
    x = Tensor(np.zeros((1, n_features)) if placeholder is None else placeholder, requires_grad=False)
    out = model(x)
    names, nodes, consts = {id(x): "input"}, [], {}
    op_name = {"MatMul": "matmul", "EAdd": "add", "ReLUOp": "relu", "BroadcastTo": "identity"}
    for t in topo_sort(out):
        if id(t) in names:
            continue
        if t.op is None:                                    # a weight: store it as a constant
            names[id(t)] = f"const{len(consts)}"
            consts[names[id(t)]] = t.cached.tolist()
            continue
        names[id(t)] = f"v{len(nodes)}"
        if type(t.op).__name__ == "AddS":                  # x + scalar: store the scalar as a constant
            c = f"const{len(consts)}"; consts[c] = float(t.op.c)
            nodes.append({"name": names[id(t)], "op": "add", "inputs": [names[id(t.inputs[0])], c]})
            continue
        nodes.append({"name": names[id(t)], "op": op_name[type(t.op).__name__],
                      "inputs": [names[id(i)] for i in t.inputs]})
    return {"nodes": nodes, "constants": consts, "output": names[id(out)]}

graph = export(model, 64)
text = json.dumps(graph)
print(f"exported {len(graph['nodes'])} nodes, {len(graph['constants'])} constants, {len(text)/1e3:.0f} KB of JSON")
for n_ in graph["nodes"]:
    print(f"  {n_['name']} = {n_['op']}({', '.join(n_['inputs'])})")

exported 7 nodes, 4 constants, 103 KB of JSON
  v0 = identity(const0)
  v1 = identity(const2)
  v2 = matmul(input, const3)
  v3 = add(v2, v1)
  v4 = relu(v3)
  v5 = matmul(v4, const1)
  v6 = add(v5, v0)


## 3. A tiny inference engine

It reads the JSON and executes nodes in order with plain NumPy — no autodiff, no `Tensor` class, nothing
from the training framework.

In [4]:
KERNELS = {"matmul": lambda a, b: a @ b, "add": lambda a, b: a + b,
           "relu": lambda a: np.maximum(0, a), "identity": lambda a: a}

def run_graph(g, x):
    values = {"input": x, **{k: np.array(v) for k, v in g["constants"].items()}}
    for n_ in g["nodes"]:
        values[n_["name"]] = KERNELS[n_["op"]](*[values[i] for i in n_["inputs"]])
    return values[g["output"]]

loaded = json.loads(text)
engine_pred = run_graph(loaded, Xte)
print("inference engine == framework:", np.allclose(engine_pred, framework_pred))

inference engine == framework: True


### Memory planning

A naive engine keeps every intermediate value alive. But once a value's last consumer has run, its buffer
can be handed to a later node of the same shape. Plan it ahead of time:

In [5]:
def plan_memory(g, batch):
    shapes = {"input": (batch, 64)}
    for k, v in g["constants"].items():
        shapes[k] = np.array(v).shape
    for n_ in g["nodes"]:                                   # infer every intermediate's shape
        ins = [shapes[i] for i in n_["inputs"]]
        shapes[n_["name"]] = (ins[0][0], ins[1][1]) if n_["op"] == "matmul" else \
                             tuple(np.broadcast_shapes(*ins)) if n_["op"] == "add" else ins[0]
    last_use = {}
    for idx, n_ in enumerate(g["nodes"]):
        for i in n_["inputs"]:
            last_use[i] = idx
    free, buffers, assign = [], 0, {}
    for idx, n_ in enumerate(g["nodes"]):
        match = next((b for b in free if b[1] == shapes[n_["name"]]), None)
        if match:
            free.remove(match); assign[n_["name"]] = match[0]
        else:
            assign[n_["name"]] = buffers; buffers += 1
        for i in n_["inputs"]:                              # inputs whose life just ended
            if i in assign and last_use.get(i) == idx:
                free.append((assign[i], shapes[i]))
    return len(g["nodes"]), buffers

deep_W = [Tensor(rng.normal(0, np.sqrt(2 / 64), (64, 64))) for _ in range(8)]
def deep_model(x):
    h = x
    for W in deep_W:
        h = relu(h @ W)                                    # 8 layers, all 64 wide
    return h

deep_graph = json.loads(json.dumps(export(deep_model, 64)))
naive, planned = plan_memory(deep_graph, batch=450)
print(f"8-layer network: {naive} intermediate arrays without planning, {planned} buffers with reuse")
naive_s, planned_s = plan_memory(loaded, batch=450)
print(f"the small trained model: {naive_s} -> {planned_s} (its intermediates are mostly different shapes)")

8-layer network: 16 intermediate arrays without planning, 2 buffers with reuse
the small trained model: 7 -> 6 (its intermediates are mostly different shapes)


## 4. Operator fusion

`matmul → add → relu` is three kernels and two temporary arrays. An inference engine fuses the pattern into one
operator that does all three in a single pass:

In [6]:
def fuse(g):
    consumers = {}
    for n_ in g["nodes"]:
        for i in n_["inputs"]:
            consumers.setdefault(i, []).append(n_)
    by_name = {n_["name"]: n_ for n_ in g["nodes"]}
    out, skip = [], set()
    for n_ in g["nodes"]:
        if n_["name"] in skip:
            continue
        nxt = consumers.get(n_["name"], [])
        if n_["op"] == "matmul" and len(nxt) == 1 and nxt[0]["op"] == "add":
            add = nxt[0]
            bias = [i for i in add["inputs"] if i != n_["name"]][0]
            bias = by_name[bias]["inputs"][0] if bias in by_name and by_name[bias]["op"] == "identity" else bias
            after = consumers.get(add["name"], [])
            if len(after) == 1 and after[0]["op"] == "relu":
                out.append({"name": after[0]["name"], "op": "dense_relu", "inputs": n_["inputs"] + [bias]})
                skip.update({add["name"], after[0]["name"]})
            else:
                out.append({"name": add["name"], "op": "dense", "inputs": n_["inputs"] + [bias]})
                skip.add(add["name"])
        elif n_["op"] != "identity":
            out.append(n_)
    return {**g, "nodes": out}

KERNELS["dense"] = lambda x, W, b: x @ W + b
KERNELS["dense_relu"] = lambda x, W, b: np.maximum(0, x @ W + b)

fused = fuse(loaded)
print(f"kernels to launch: {len(loaded['nodes'])} -> {len(fused['nodes'])}")
for n_ in fused["nodes"]:
    print(f"  {n_['name']} = {n_['op']}({', '.join(n_['inputs'])})")
print("fused graph gives the same answer:", np.allclose(run_graph(fused, Xte), framework_pred))

kernels to launch: 7 -> 2
  v4 = dense_relu(input, const3, const2)
  v6 = dense(v4, const1, const0)
fused graph gives the same answer: True


## 5. What a traced graph cannot capture

Exporting by running the model once records only the path that run took. A model with ordinary Python control
flow — like stochastic depth, or anything that branches on the data — loses its other branches:

In [7]:
def model_with_if(x):
    h = x @ W1
    h = relu(h + broadcast_to(b1, h.shape))
    if float(x.cached.sum()) > 10:                         # Python decides, at trace time
        h = relu(h + (-1.0))                               # an extra thresholding step on bright inputs
    out = h @ W2
    return out + broadcast_to(b2, out.shape)

g_dark = export(model_with_if, 64)                         # traced with an all-zero input
g_bright = export(model_with_if, 64, placeholder=np.ones((1, 64)))   # traced with a bright input
print("nodes recorded: dark trace", len(g_dark["nodes"]), "| bright trace", len(g_bright["nodes"]))

bright = Xte[Xte.sum(1) > 10]
true_out = model_with_if(Tensor(bright, requires_grad=False)).cached
dark_out = run_graph(g_dark, bright)
print(f"bright test inputs: {len(bright)}")
print(f"largest logit error of the dark-traced graph: {np.abs(true_out - dark_out).max():.2f}")
print(f"predictions that disagree with the framework: {np.mean(true_out.argmax(1) != dark_out.argmax(1)):.1%}")

nodes recorded: dark trace 7 | bright trace 9
bright test inputs: 450
largest logit error of the dark-traced graph: 7.28
predictions that disagree with the framework: 2.0%


Tracing with a dark placeholder never took the branch, so the thresholding step is simply missing — two fewer
nodes — and on real bright inputs that graph computes something different from the framework. This is the tension the lecture describes: Python's flexibility makes models easy to build
and hard to export, so deployment formats add explicit control-flow nodes, or such code must be rewritten.

## Summary

- Exporting walks the graph once and writes plain data; a NumPy-only engine reproduced the framework exactly.
- Liveness-based planning ran an 8-layer network with 2 buffers instead of 16 arrays.
- Fusing `matmul → add → relu` cut the number of kernels and still gave identical answers.
- Tracing records one path through the code: data-dependent `if`s are silently dropped.